# 01 · Data collection

**Project:** Indian Consumer Price & Market Intelligence System
**Notebook goal:** get real consumer price data from sources that allow it, and keep an untouched copy of everything I download.

Before writing any code I spent time checking *where* Indian retail prices can be collected legitimately. The quick answer: most of the obvious places (BigBasket, Blinkit, Amazon, JioMart…) forbid scraping in their terms, and the government's centre-wise price reports sit behind a CAPTCHA. So I stuck to three sources that are open by design:

| Source | What I get from it | How |
|---|---|---|
| **DoCA Price Monitoring Division** (Dept. of Consumer Affairs) | Official All-India average retail price of ~42 essentials, plus the official value 1 week, 1 month, 6 months, 1 year and 2 years earlier | public dashboard page, plain HTTP GET |
| **Open Prices** (Open Food Facts) | Individual shop prices of branded products — brand, pack size, discount, city, date | public JSON API |
| **MoSPI CPI** | Official monthly CPI index — used **only as a benchmark**, never mixed with prices | public JSON API |

What I deliberately did **not** use is listed in `docs/SOURCES.md` together with the reason.

## Setup

In [1]:
import sys, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 45)

## Collect a new snapshot (optional)

`extract.py` writes one new file per source into `data/raw/<source>/<date>/`. Every line of that file is the server's response exactly as received (wrapped with the URL, time and HTTP status), so the raw files can always be traced back to the source.

The first snapshot (1 Oct 2026) was captured through my browser because my development machine's network blocks these sites. On a normal connection, flip the switch below to `True` to collect today's snapshot.

In [2]:
RUN_EXTRACT = False      # set True to download a new snapshot (needs internet access to the three sites)

if RUN_EXTRACT:
    import extract
    print(extract.extract_open_prices())
    print(extract.extract_doca())

## What is in the raw folder?

The SHA-256 hash is the file's fingerprint. The loader remembers it, so if a raw file is ever edited by mistake the pipeline stops instead of silently using changed data.

In [3]:
import hashlib

rows = []
for p in sorted((ROOT / "data" / "raw").rglob("*.jsonl")):
    rows.append({
        "source": p.parts[-3],
        "capture_date": p.parts[-2],
        "file": p.name,
        "size_kb": round(p.stat().st_size / 1024, 1),
        "lines": sum(1 for _ in open(p, encoding="utf-8")),
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16] + "…",
    })
raw_inventory = pd.DataFrame(rows)
raw_inventory

,source,capture_date,file,size_kb,lines,sha256
0,doca_dashboard,2026-10-01,doca_dashboard_raw_2026-10-01.jsonl,123.5,2,88bd44fefe06e302…
1,mospi_cpi,2026-10-01,mospi_cpi_raw_2026-10-01.jsonl,1856.2,920,0d4435dcb18cc7b1…
2,mospi_cpi,2026-10-01,mospi_cpi_raw_2026-10-01_backfill.jsonl,197.5,78,ace37019545800d2…
3,open_prices,2026-10-01,openprices_INR_raw_2026-10-01.jsonl,1169.2,5,1bb2ed0abcf53c56…


## A first look at each source

### Open Prices
Each line of the file is one API page of up to 100 price records. The currency filter was `INR`; a handful of INR prices turn out to be recorded outside India — I keep them in the raw file and filter them out later (with a logged reason).

In [4]:
import json

op_file = next((ROOT / "data/raw/open_prices").rglob("*.jsonl"))
items = []
for line in open(op_file, encoding="utf-8"):
    page = json.loads(line)
    page = json.loads(page["body"]) if "body" in page else page
    items.extend(page["items"])

print(f"{len(items)} price records in the raw file")
countries = pd.Series([(i.get("location") or {}).get("osm_address_country") for i in items]).value_counts(dropna=False)
countries

402 price records in the raw file


India            390
Pilipinas          4
Gabon              3
NaN                2
Nederland          1
United States      1
Philippines        1
Name: count, dtype: int64

In [5]:
# one record, flattened to the fields this project cares about
ex = items[5]
pd.Series({
    "price id": ex["id"],
    "date": ex["date"],
    "price (INR)": ex["price"],
    "discounted?": ex["price_is_discounted"],
    "price before discount": ex["price_without_discount"],
    "barcode": ex["product_code"],
    "product": ex["product"]["product_name"],
    "brand": ex["product"]["brands"],
    "pack size": f'{ex["product"]["product_quantity"]} {ex["product"]["product_quantity_unit"]}',
    "shop / place": ex["location"]["osm_display_name"],
})

price id                                                        31312
date                                                       2024-08-12
price (INR)                                                      40.0
discounted?                                                     False
price before discount                                            None
barcode                                                 8906059639890
product                                          Chocolate Milk Shake
brand                                                        Epigamia
pack size                                                      180 ml
shop / place             Powai, S Ward, Mumbai Zone 6, Mumbai, Mum...
dtype: object

### DoCA dashboard
The dashboard is an HTML table. The second and third header rows tell me what each column means (`Price As On`, `1 Week back`, …) and which date it refers to. Values in look-back columns carry the % change in brackets, e.g. `46.53 (-0.21)`; `--` means no price was reported.

In [6]:
import io

doca_file = next((ROOT / "data/raw/doca_dashboard").rglob("*.jsonl"))
pages = [json.loads(l) for l in open(doca_file, encoding="utf-8")]
for p in pages:
    print(p["http_status"], p["fetched_at_utc"], p["url"])

dash_html = next(p["body"] for p in pages if "DBprices" in p["url"])
doca_table = max(pd.read_html(io.StringIO(dash_html)), key=len)
doca_table.head(8)

200 2026-10-01T05:07:50.676Z https://fcainfoweb.nic.in/Reports/DB/DBprices.aspx
200 2026-10-01T05:07:53.924Z https://fcainfoweb.nic.in/


,0,1,2,3,4,5,6,7
0,Commodities,Unit,Daily Average Retail Price,Daily Average Retail Price,Daily Average Retail Price,Daily Average Retail Price,Daily Average Retail Price,Daily Average Retail Price
1,Commodities,Unit,Price As On,1 Week back,1 Month back,6 Months back,1 Year back,2 Years back
2,Commodities,Unit,30/09/2026,23/09/2026,30/08/2026,30/03/2026,30/09/2025,30/09/2024
3,Broken Rice,1 Kg,33.84,34.05 (-0.62),32.56 (3.93),--,--,--
4,Rice,1 Kg,46.43,46.53 (-0.21),45.53 (1.98),43.11 (7.70),42.92 (8.18),43.42 (6.93)
5,Wheat,1 Kg,31.88,31.86 (0.06),31.8 (0.25),31.26 (1.98),31.89 (-0.03),31.11 (2.48)
6,Atta (Wheat),1 Kg,37.54,37.66 (-0.32),37.45 (0.24),37 (1.46),37.25 (0.78),36.01 (4.25)
7,Gram Dal(Bharat Dal Adjusted),1 Kg,84.61,84.13 (0.57),82.17 (2.97),81.73 (3.52),82.77 (2.22),88.58 (-4.48)


### MoSPI CPI
Without registering, the API returns 10 rows per page, so the capture is ~1,000 small pages. While checking the data I noticed that big paginated requests are **not stably ordered**: some rows came back twice and others never came back. The second (backfill) file re-requests just the affected month × sector slices, which are small enough to come back complete. Notebook 04 shows the numbers.

In [7]:
cpi_rows = []
for f in sorted((ROOT / "data/raw/mospi_cpi").rglob("*.jsonl")):
    for line in open(f, encoding="utf-8"):
        env = json.loads(line)
        for d in json.loads(env["body"])["data"]:
            cpi_rows.append({"file": f.name, "kind": env.get("kind"), **d})
cpi_raw = pd.DataFrame(cpi_rows)
print(len(cpi_raw), "rows across", cpi_raw["file"].nunique(), "files")
cpi_raw.groupby(["file", "kind"]).size().rename("rows")

9920 rows across 2 files


file                                     kind                   
mospi_cpi_raw_2026-10-01.jsonl           group_allindia             2016
                                         item                       7176
mospi_cpi_raw_2026-10-01_backfill.jsonl  group_allindia_backfill     728
Name: rows, dtype: int64

## Takeaways

- Three legitimate sources, all captured without logins, CAPTCHAs or anti-bot workarounds.
- Raw files are saved exactly as received and fingerprinted.
- The official DoCA history in the first snapshot is **six dated points per commodity** (latest, −1 week, −1 month, −6 months, −1 year, −2 years). Anything finer has to be built up by running the extractor regularly — I don't fill gaps with made-up values.

Next: **02 · Cleaning and standardisation**.